# 02 — Data Preprocessing and Feature Engineering

**Personal Tabular Machine Learning Handbook**


## Learning objectives
- Split data before learning preparation rules and explain data leakage.
- Choose missing-value handling, encoding, scaling, outlier checks, and features appropriately.
- Build a ColumnTransformer/Pipeline and save typed, traceable splits for modeling.

**Primary project:** Predict NusantaraMart monthly sales from region and promotional spending. All techniques are references; not every technique should be applied to this small dataset.

**Before you run:** Run from the repository root. Notebook 01 provides the reasoning; this chapter can construct its artifacts directly from the raw CSV. Run this before 03 and 04.

**How to use this notebook:** Read top to bottom the first time. Later, search for a `Technique:` heading or use the Quick Syntax Reference. Each code example is executable after its preceding setup cells. Outputs are included for revision.

## A. Introduction to Preprocessing

**Features (X)** are inputs available when predicting. The **target (y)** is the outcome to predict. Numerical and categorical inputs need different preparation. IDs are retained for tracking, not modeling.

- **Training:** Fit parameters, preprocessing statistics, and cross-validation candidates.
- **Validation:** Compare choices without touching the test set.
- **Test:** Estimate performance after all model and feature choices are fixed.

**Data leakage** occurs when information unavailable at prediction time influences training. Examples: scaling with test-set means, imputing with the full dataset, selecting features using held-out targets, or using future sales to predict current sales. Leakage makes evaluation overoptimistic.

Most transformations learn something: a median, category list, or scaling range. **Split first, fit on training only, transform other splits using the same fitted rules.** Later, after model selection, a final pipeline can be refitted on training plus validation before one test evaluation.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression

raw = pd.read_csv("data/raw/nusantaramart_simple_sales.csv", dtype={"store_id": "string", "region": "string"})
features = ["promo_spend_million", "region"]
target = "monthly_sales_million"

## B. Train-Test Splitting

### Technique: train_test_split, proportions, and random states

**What is it?** A split assigns independent observations to distinct development and evaluation roles.

**Why do we use it?** Generalization must be measured on observations not used to fit or choose the model.

**When should we use it?** Random row splits suit independent, exchangeable stores. Use time/group splits for dependent records.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [2]:
development, test = train_test_split(raw, test_size=0.20, random_state=42)
train, validation = train_test_split(development, test_size=0.25, random_state=42)
X_train, y_train = train[features].copy(), train[target].copy()
X_validation, y_validation = validation[features].copy(), validation[target].copy()
X_test, y_test = test[features].copy(), test[target].copy()
print("Train / validation / test:", len(train), len(validation), len(test))
assert set(train.store_id).isdisjoint(validation.store_id)
assert set(train.store_id).isdisjoint(test.store_id)
assert set(validation.store_id).isdisjoint(test.store_id)

Train / validation / test: 51 17 18


**Interpretation:** 20% is reserved for testing, then 25% of the remaining 80% for validation: approximately 60/20/20. The seed reproduces assignment with the same data order.

**Things to Remember:** A seed is not a guarantee of representativeness. Keep test outcomes closed below. Stratifying a continuous target is not the same as classification stratification.

### Technique: Stratified classification splits

**What is it?** Stratification approximately preserves class proportions in each split.

**Why do we use it?** Rare classes may otherwise vanish from a small validation set.

**When should we use it?** Use with a categorical classification target and enough examples per class.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [3]:
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer(as_frame=True)
class_X = cancer.data
class_y = (cancer.target == 0).astype(int)  # 1 = malignant; 0 = benign
class_X_train, class_X_holdout, class_y_train, class_y_holdout = train_test_split(
    class_X, class_y, test_size=0.20, stratify=class_y, random_state=42
)
print("Training class proportions:")
print(class_y_train.value_counts(normalize=True).sort_index())

Training class proportions:
target
0    0.626374
1    0.373626
Name: proportion, dtype: float64


**Interpretation:** The example uses a built-in, offline dataset with numerical cell measurements. We explicitly define malignant as the positive class. The holdout is not evaluated here.

**Things to Remember:** Stratification does not solve class imbalance or dependence. This classification example is educational, not a clinical decision tool.

## C. Handling Missing Values

The supplied primary data has no missing entries under our checks. The following small table deliberately contains missing values to teach behavior; it is separate from the client data and is never used for primary model scores.

In [4]:
missing_train = pd.DataFrame({"amount": [10.0, 12.0, np.nan, 50.0], "visits": [1.0, 2.0, 3.0, 4.0]})
missing_new = pd.DataFrame({"amount": [np.nan, 20.0], "visits": [2.0, 3.0]})
category_train = pd.DataFrame({"region": ["Java", np.nan, "Java", "Sumatra"]})
print("Primary training missingness:", X_train.isna().sum().to_dict())
display(missing_train)

Primary training missingness: {'promo_spend_million': 0, 'region': 0}


,amount,visits
0,10.0,1.0
1,12.0,2.0
2,NaN,3.0
3,50.0,4.0


### Technique: Detection and dropping missing observations

**What is it?** isna locates missing entries; dropna removes selected incomplete observations.

**Why do we use it?** Dropping may be reasonable if very few records are incomplete for an understood reason.

**When should we use it?** Consider dropping when loss is small and unlikely to bias the sample; investigate missingness first.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [5]:
print(missing_train.isna().sum())
print(missing_train.isna().mean())
complete = missing_train.dropna(subset=["amount"])
display(complete)

amount    1
visits    0
dtype: int64
amount    0.25
visits    0.00
dtype: float64


,amount,visits
0,10.0,1.0
1,12.0,2.0
3,50.0,4.0


**Interpretation:** The example has one missing amount. Removing it leaves three records and changes the sample.

**Things to Remember:** Do not drop all rows indiscriminately. Missingness may be systematic; an unlabeled target normally cannot be used for supervised training.

### Technique: Mean, median, mode, and SimpleImputer

**What is it?** Imputation replaces missing values with a statistic learned from training data.

**Why do we use it?** Many models require complete inputs; a fitted imputer supplies a consistent rule at prediction time.

**When should we use it?** Mean suits roughly symmetric numeric features; median is less sensitive to extremes; most_frequent works for categories but can overrepresent the common class.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [6]:
for strategy in ["mean", "median"]:
    imputer = SimpleImputer(strategy=strategy)
    imputer.fit(missing_train[["amount"]])
    print(strategy, "training statistic:", imputer.statistics_, "new values:", imputer.transform(missing_new[["amount"]]).ravel())
mode_imputer = SimpleImputer(strategy="most_frequent")
print(mode_imputer.fit_transform(category_train).ravel())

mean training statistic: [24.] new values: [24. 20.]
median training statistic: [12.] new values: [12. 20.]
['Java' 'Java' 'Java' 'Sumatra']


**Interpretation:** Both numeric methods learn only from missing_train; missing_new does not affect the statistic. The extreme value 50 makes the mean larger than the median. The categorical gap becomes Java.

**Things to Remember:** Imputation is an estimate and can reduce variability. A missingness indicator (add_indicator=True) can help when absence itself carries information, but validate it.

### Technique: KNNImputer introduction

**What is it?** K-nearest-neighbor imputation estimates a missing value from similar observed rows.

**Why do we use it?** Relationships between features can improve on a single global fill value.

**When should we use it?** Consider it when features describe meaningful similarity and the dataset is modest; compare against a simple baseline.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [7]:
# Scale first so a large-unit feature does not dominate distance.
knn_steps = Pipeline([
    ("scale", StandardScaler()),
    ("impute", KNNImputer(n_neighbors=2)),
])
knn_steps.fit(missing_train)
display(pd.DataFrame(knn_steps.transform(missing_new), columns=missing_train.columns))

,amount,visits
0,-0.706410,-0.447214
1,-0.217357,0.447214


**Interpretation:** The output is in standardized units because the pipeline scales before imputing. The neighbors and scaling are learned only from the teaching training table.

**Things to Remember:** KNN can be slow, sensitive to scaling, and weak in high dimensions. StandardScaler supports NaNs here, but not every transformer does. Do not mix arbitrary category codes into a distance calculation.

## D. Categorical Encoding

### Technique: LabelEncoder for target labels

**What is it?** Label encoding maps target class names to integers.

**Why do we use it?** Classification estimators often accept integers or labels; a saved mapping clarifies predictions.

**When should we use it?** Use LabelEncoder for y, not unordered feature columns.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [8]:
label_encoder = LabelEncoder()
encoded_target = label_encoder.fit_transform(["benign", "malignant", "benign"])
print(label_encoder.classes_, encoded_target)
print(label_encoder.inverse_transform([1, 0]))

['benign' 'malignant'] [0 1 0]
['malignant' 'benign']


**Interpretation:** The classes are assigned integer IDs; inverse_transform restores names. In our cancer workflow we use the explicit 1=malignant mapping instead.

**Things to Remember:** Integer target labels do not imply a numerical distance between classes. Unknown target labels indicate a changed classification problem.

### Technique: OrdinalEncoder for meaningful order

**What is it?** Ordinal encoding represents categories with a known ranking.

**Why do we use it?** It preserves order such as low, medium, high without creating many columns.

**When should we use it?** Use when the order is real and appropriate for the model.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [9]:
priority_train = pd.DataFrame({"priority": ["low", "high", "medium"]})
ordinal = OrdinalEncoder(categories=[["low", "medium", "high"]],
                         handle_unknown="use_encoded_value", unknown_value=-1)
print(ordinal.fit_transform(priority_train).ravel())
print(ordinal.transform(pd.DataFrame({"priority": ["urgent", "low"]})).ravel())

[0. 2. 1.]
[-1.  0.]


**Interpretation:** low, medium, high map to 0, 1, 2; an unseen label maps to -1 under the chosen rule.

**Things to Remember:** A linear model interprets spacing between codes numerically; ranked categories may not have equal spacing. Java and Sumatra have no natural order.

### Technique: OneHotEncoder and unseen categories

**What is it?** One-hot encoding creates a separate indicator for each category.

**Why do we use it?** It represents nominal features without inventing an order.

**When should we use it?** Use for modest-cardinality categories with linear models or standard scikit-learn estimators.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [10]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
encoded_region = encoder.fit_transform(X_train[["region"]])
print(encoder.get_feature_names_out())
display(pd.DataFrame(encoded_region, columns=encoder.get_feature_names_out(), index=X_train.index).head())
print("Known category example:", encoder.transform(pd.DataFrame({"region": ["Java"]})))

['region_Sumatra']


,region_Sumatra
65,1.0
27,0.0
6,1.0
43,1.0
82,0.0


Known category example: [[0.]]


**Interpretation:** With two regions and drop="first", one indicator remains; the omitted category is the reference. The fitted category list comes only from training.

**Things to Remember:** handle_unknown="ignore" keeps future categories from crashing but encodes them as all zeros; with a dropped reference they are indistinguishable from it. Monitor unexpected categories. High-cardinality IDs should not be blindly one-hot encoded.

## E. Feature Scaling

### Technique: StandardScaler, MinMaxScaler, RobustScaler; fit_transform versus transform

**What is it?** Scaling changes numeric units. StandardScaler centers/scales by mean/std; MinMaxScaler uses the training range; RobustScaler uses median/IQR.

**Why do we use it?** Distance-based and regularized linear models are sensitive to feature scales.

**When should we use it?** Use standard scaling for many linear/logistic models, min–max for desired training ranges, and robust scaling for skewed/extreme numeric inputs.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [11]:
for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    scaled_train = scaler.fit_transform(X_train[["promo_spend_million"]])
    scaled_validation = scaler.transform(X_validation[["promo_spend_million"]])
    print(type(scaler).__name__, "training range:", scaled_train.min().round(2), scaled_train.max().round(2))
    print("First validation transform:", scaled_validation[0].round(2))

StandardScaler training range: -1.7 1.64
First validation transform: [-1.03]
MinMaxScaler training range: 0.0 1.0
First validation transform: [0.2]
RobustScaler training range: -1.19 0.94
First validation transform: [-0.77]


**Interpretation:** fit_transform learns statistics then transforms training rows. transform applies those unchanged statistics to validation. Min–max validation values may lie outside [0, 1] if outside the training range.

**Things to Remember:** Never fit a new scaler to validation/test data. Tree splits usually do not need scaling. Unregularized linear regression can fit without it, but scaling assists coefficient comparison and many regularized/optimization-based models. Scaling does not remove outliers.

## F. Outlier Handling

### Technique: IQR bounds and Z-scores learned on training data

**What is it?** IQR measures central spread; Z-scores express distance from the training mean in training standard deviations.

**Why do we use it?** They flag inputs worth review and help distinguish errors from legitimate extremes.

**When should we use it?** Use domain rules first; Z-score heuristics are more useful with roughly bell-shaped features.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [12]:
spending = X_train["promo_spend_million"]
q1, q3 = spending.quantile([0.25, 0.75])
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
iqr_flags = (spending < lower) | (spending > upper)
z_scores = (spending - spending.mean()) / spending.std(ddof=0)
print("IQR / absolute Z > 3 flags:", iqr_flags.sum(), (z_scores.abs() > 3).sum())
validation_flags = (X_validation["promo_spend_million"] < lower) | (X_validation["promo_spend_million"] > upper)
print("Validation flags using training bounds:", validation_flags.sum())

IQR / absolute Z > 3 flags: 0 0
Validation flags using training bounds: 0


**Interpretation:** Counts report only heuristic review candidates. These examples keep all records; valid high-spending stores remain part of the prediction problem.

**Things to Remember:** Do not delete legitimate extreme customers or test errors just to raise a score. Z-scores are undefined for zero-variance columns. If clipping is justified, learn bounds inside each CV fold, not before CV.

## G. Feature Engineering

Feature engineering creates new inputs; feature selection chooses among existing inputs. A feature must be available at prediction time and must not use the outcome it is intended to predict.

### Technique: Ratios, differences, interactions, dates, bins, and log transforms

**What is it?** Engineered features express domain relationships more explicitly.

**Why do we use it?** A useful representation can make a simple model more effective.

**When should we use it?** Use when the business meaning is defensible, the necessary inputs exist at prediction time, and validation supports the change.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [13]:
# Constructed teaching inputs: NOT additional NusantaraMart measurements.
engineering = pd.DataFrame({"spend": [0., 10., 20.], "visits": [5., 20., 40.],
                            "previous_spend": [0., 8., 15.],
                            "opened": pd.to_datetime(["2020-01-01", "2022-07-01", "2024-01-01"])})
engineering["spend_per_visit"] = engineering["spend"] / engineering["visits"].replace(0, np.nan)
engineering["spend_change"] = engineering["spend"] - engineering["previous_spend"]
engineering["spend_x_visits"] = engineering["spend"] * engineering["visits"]
engineering["store_age_days"] = (pd.Timestamp("2025-01-01") - engineering["opened"]).dt.days
engineering["spend_band"] = pd.cut(engineering["spend"], bins=[-np.inf, 10, np.inf], labels=["low", "high"])
engineering["log_spend"] = np.log1p(engineering["spend"])
display(engineering)

,spend,visits,previous_spend,opened,spend_per_visit,spend_change,spend_x_visits,store_age_days,spend_band,log_spend
0,0.0,5.0,0.0,2020-01-01,0.0,0.0,0.0,1827,low,0.000000
1,10.0,20.0,8.0,2022-07-01,0.5,2.0,200.0,915,low,2.397895
2,20.0,40.0,15.0,2024-01-01,0.5,5.0,800.0,366,high,3.044522


**Interpretation:** Ratios normalize scale, differences capture change, interactions capture combined effects, dates give age, bins create categories, and log1p compresses nonnegative values while accepting zero. The fixed cutoff date makes age reproducible.

**Things to Remember:** Never use sales/spend to predict the same sales target: it leaks y. log1p is unsuitable for values ≤−1 and needs domain justification. This example uses predetermined bins; data-derived quantiles must be fitted on training folds. Any learned feature step belongs in the pipeline.

## H. Feature Selection

Our actual two-feature workflow retains both predictors. The following alternatives demonstrate selection syntax; they are not stacked together into the final model.

### Technique: Correlation filtering and VarianceThreshold

**What is it?** Correlation filtering identifies redundant numerical predictors; VarianceThreshold removes near-constant columns.

**Why do we use it?** Redundant or constant inputs can add complexity without useful signal.

**When should we use it?** Fit selections on training folds only and preserve the resulting column list for later transforms.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [14]:
selection_demo = pd.DataFrame({"spend": X_train["promo_spend_million"],
                               "spend_copy": X_train["promo_spend_million"] * 2,
                               "constant": 1.0})
absolute_corr = selection_demo.corr().abs()
upper_triangle = absolute_corr.where(np.triu(np.ones(absolute_corr.shape), k=1).astype(bool))
to_drop = [col for col in upper_triangle.columns if (upper_triangle[col] > 0.95).any()]
print("Redundant columns in constructed demo:", to_drop)
variance_filter = VarianceThreshold(threshold=0.0).fit(selection_demo)
print("Nonconstant columns:", variance_filter.get_feature_names_out())

Redundant columns in constructed demo: ['spend_copy']
Nonconstant columns: ['spend' 'spend_copy']


**Interpretation:** The deliberately duplicated spending feature is redundant; the constant has no variation. NaN correlations for the constant are expected.

**Things to Remember:** A low marginal correlation does not rule out nonlinear or interaction value. High correlation alone is not a universal reason to drop a predictor, especially in trees. A threshold depends on scale; these are training-only teaching examples.

### Technique: SelectKBest and training-based feature importance

**What is it?** SelectKBest ranks each feature against y; a fitted tree can provide an alternative importance ranking.

**Why do we use it?** They offer ways to reduce a large input space, with different assumptions.

**When should we use it?** Treat rankings as candidates for validation, not causal explanations.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [15]:
from sklearn.ensemble import RandomForestRegressor
selector = SelectKBest(score_func=f_regression, k=1)
selector.fit(X_train[["promo_spend_million"]], y_train)
print("Selected:", selector.get_feature_names_out())
importance_demo = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
importance_demo.fit(X_train[["promo_spend_million"]], y_train)
print("Single-feature tree importance:", importance_demo.feature_importances_)

Selected: ['promo_spend_million']
Single-feature tree importance: [1.]


**Interpretation:** With only one numerical input, it is necessarily retained and gets all tree importance. This illustrates syntax, not proof that region or other absent drivers have no value.

**Things to Remember:** f_regression checks linear marginal association. Select features inside CV, not once on all development data before CV. Tree impurity importance can favor high-cardinality inputs; Notebook 04 introduces held-out permutation importance.

## I. Preprocessing Pipelines

### Technique: Pipeline and ColumnTransformer

**What is it?** A Pipeline chains steps; ColumnTransformer applies different steps to named subsets of columns.

**Why do we use it?** They keep training, inference, and cross-validation consistent and fit each learned rule at the correct time.

**When should we use it?** Use a fresh complete pipeline per model/CV candidate rather than scaling the entire dataset in advance.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [16]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
])
preprocessor_template = ColumnTransformer([
    ("numeric", numeric_pipeline, ["promo_spend_million"]),
    ("categorical", categorical_pipeline, ["region"]),
], remainder="drop", verbose_feature_names_out=True)
preprocessor_template

,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


**Interpretation:** One numeric column is imputed and standardized; region is imputed and one-hot encoded. Named selectors preserve intended behavior even if input DataFrame order changes.

**Things to Remember:** The template is intentionally unfitted. During CV, the entire pipeline must be fitted separately inside each fold. Converting everything to arrays too early can lose column labels.

### Technique: Fit only training; transform validation and test

**What is it?** A fitted transformer stores training statistics and category definitions.

**Why do we use it?** Consistent transformations prevent leakage and mismatched columns.

**When should we use it?** Fit once on training for a preparation demonstration; refit fresh clones inside CV in Notebook 03.

**Python Syntax / Example**

The following cell is a runnable example; adapt the column names and values to reuse it.

In [17]:
fitted_preprocessor = clone(preprocessor_template)
train_array = fitted_preprocessor.fit_transform(X_train, y_train)
validation_array = fitted_preprocessor.transform(X_validation)
test_array = fitted_preprocessor.transform(X_test)  # no test outcomes used
feature_names = fitted_preprocessor.get_feature_names_out().tolist()
train_transformed = pd.DataFrame(train_array, columns=feature_names, index=X_train.index)
display(train_transformed.head())
print("Transformed shapes:", train_array.shape, validation_array.shape, test_array.shape)

,numeric__promo_spend_million,categorical__region_Sumatra
65,-0.663475,1.0
27,-1.342603,0.0
6,1.564540,1.0
43,0.182456,1.0
82,-1.378346,0.0


Transformed shapes: (51, 2) (17, 2) (18, 2)


**Interpretation:** The three arrays share feature order and width. Test inputs can be transformed by fixed training rules without fitting or inspecting test outcomes; no score is computed here.

**Things to Remember:** These pretransformed arrays are useful for inspection, not for CV model selection: fitting preprocessing before CV would let fold-validation inputs influence preprocessing. Notebook 03 loads raw splits and clones the unfitted template instead.

## J. Preparing Data for Modeling

Joblib preserves Pandas column names, order, indices, and dtypes. The bundle below is the authoritative handoff. CSV exports are convenient previews, not substitutes for typed artifacts. Load serialized artifacts only from trusted sources and use compatible library versions.

In [18]:
processed = Path("data/processed")
processed.mkdir(parents=True, exist_ok=True)
bundle = {
    "X_train": X_train, "y_train": y_train,
    "X_validation": X_validation, "y_validation": y_validation,
    "X_test": X_test, "y_test": y_test,
    "train_ids": train.store_id.copy(), "validation_ids": validation.store_id.copy(), "test_ids": test.store_id.copy(),
    "feature_order": features, "target": target, "seed": 42,
    "preprocessor_template": preprocessor_template,
}
joblib.dump(bundle, processed / "regression_splits.joblib")
joblib.dump(fitted_preprocessor, processed / "train_fitted_preprocessor.joblib")
train_transformed.to_csv(processed / "training_transformed_preview.csv", index=True, index_label="source_row")
schema = {"raw_feature_order": features, "raw_dtypes": X_train.dtypes.astype(str).to_dict(),
          "transformed_feature_order": feature_names, "target": target, "units": "IDR million",
          "split_sizes": [len(train), len(validation), len(test)], "sklearn_version": sklearn.__version__}
(processed / "schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
print("Saved typed splits, fitted training transformer, preview, and schema.")

Saved typed splits, fitted training transformer, preview, and schema.


**Interpretation:** Notebook 03 will use `regression_splits.joblib` and its **unfitted** preprocessing template to train leakage-safe pipelines. Notebook 04 will use the same held-out rows. The fitted standalone transformer is a reference artifact; the final saved model will include its own fitted preprocessing.

## Summary and Quick Syntax Reference

| Need | Syntax / rule |
|---|---|
| Reproducible split | `train_test_split(X, y, test_size=.2, random_state=42)` |
| Classification proportions | `train_test_split(X, y, stratify=y, random_state=42)` |
| Median / mode fill | `SimpleImputer(strategy="median")` / `strategy="most_frequent"` |
| Neighbor fill | `KNNImputer(n_neighbors=3)`; scale meaningfully first |
| Target encoding | `LabelEncoder().fit_transform(y)` |
| Nominal encoding | `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` |
| Ordered encoding | `OrdinalEncoder(categories=[["low", "high"]])` |
| Standard / range / robust scale | `StandardScaler()` / `MinMaxScaler()` / `RobustScaler()` |
| Correct fitting | `fit_transform(X_train)` then `transform(X_validation)` |
| Feature creation | `df["interaction"] = df["a"] * df["b"]` |
| Feature selection | `SelectKBest(f_regression, k=1)`; inside CV pipeline |
| Different column rules | `ColumnTransformer([(name, transformer, columns)])` |
| Reusable workflow | `Pipeline([("prepare", prep), ("model", estimator)])` |
| Typed artifact | `joblib.dump(bundle, path)` / `joblib.load(path)` |

## Practice exercises
1. Replace the numeric scaler with `RobustScaler` in a cloned preprocessing template. Fit only on training and transform validation. **Check:** transformed column count and row order are unchanged.
2. Transform a new DataFrame containing a missing spending value and a known region using `fitted_preprocessor`. **Check:** the result is finite; the training median, not the new batch median, is used.
3. Propose one feature that would require additional data (for example, store age), and one that would leak the target. Explain when each would be available.

**Common mistakes:** fitting on all rows, transforming with a new unfitted encoder, selecting features outside CV, deleting valid extremes, giving unordered regions arbitrary ranks, and losing schema information when exporting CSV.